# Task 2 — Auto Loader: landing -> bronze (Delta)
Read JSON files from landing and loads them to Delta table in `gabrielajaniszews786_bronze`.
Idempotent: the checkpoint tracks which files were already loaded, so re-runs don't duplicate data.
Adds metadata columns (source filename, ingestion timestamp, load date). Trigger `availableNow` processes all available files, then stops (it doesn't run 24/7).

In [0]:
from pyspark.sql.functions import current_timestamp, current_date

CATALOG    = "dbr_dev"
STORAGE_ACCOUNT = "dlspl21databricks"
CONTAINER  = "gabrielajaniszews786"
SCHEMA     = "gabrielajaniszews786_bronze"
BASE       = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
LANDING    = f"/Volumes/{CATALOG}/{SCHEMA}/entsoe_landing"
SCHEMA_LOC = f"{BASE}/_schema/entsoe_prices" # Schema location
CHECKPOINT = f"{BASE}/_checkpoint/entsoe_prices" # Checkpoint
TARGET     = f"{CATALOG}.{SCHEMA}.entsoe_prices"

In [0]:
df = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "json")
      .option("cloudFiles.schemaLocation", SCHEMA_LOC)
      .option("cloudFiles.inferColumnTypes", "true")   # To prevent prices from being read as strings
      .load(LANDING)
      # --- Metadata columns required in the task ---
      .selectExpr("*",
                  "_metadata.file_name as source_file",         # source filename
                  "_metadata.file_path as source_path")
      .withColumn("ingestion_ts", current_timestamp())          # ingestion timestamp
      .withColumn("load_date",    current_date()))              # load date

In [0]:
(df.writeStream
   .format("delta")
   .option("checkpointLocation", CHECKPOINT)   # Indempotency: tracks processed files
   .trigger(availableNow=True)                 # Process outstanding files
   .toTable(TARGET))

In [0]:
# Verification check
cnt = spark.table(TARGET).count()
assert cnt > 0, "Bronze empty - ingestion problem"

In [0]:
# Idempotency test

spark.table(TARGET).count()